# Batch Eelbrain-Main Pipeline: Gammatone-8

Use this notebook to inspect and optionally run the Eelbrain-main `gammatone-8` model across subjects. By default it does not run the batch.


In [ ]:
from pathlib import Path
import sys

def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / 'analysis' / 'trf_pipeline']
    for path in candidates:
        if (path / 'alice_eelbrain_main_experiment.py').exists():
            return path
    raise FileNotFoundError(f'Could not find alice_eelbrain_main_experiment.py from {start}')


PIPELINE_DIR = find_pipeline_dir()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from alice_eelbrain_main_experiment import TRF_OPTIONS, alice

MODEL = 'gammatone-8'
RUN_BATCH = True
STATE = {'raw': '0.5-20', 'epoch': 'story-segments', 'inv': ''}

print(f'Pipeline directory: {PIPELINE_DIR}')
print(f'RUN_BATCH: {RUN_BATCH}')


## Inspect Batch Definition

This confirms what the batch job will run before launching any computation.

In [ ]:
subjects = alice.get_field_values('subject')
print(f'Subjects: {len(subjects)}')
print(subjects[:5], '...', subjects[-5:])
print('Model:', MODEL)
print('TRF options:')
for key, value in TRF_OPTIONS.items():
    print(f'  {key}: {value}')


## Preview Cache Targets

Eelbrain main writes results to its pipeline cache. This cell shows expected cache paths for a few subjects without fitting the model.


In [ ]:
for subject in subjects[:5]:
    path = alice.load_trf(MODEL, subject=subject, path_only=True, **STATE, **TRF_OPTIONS)
    print(subject, path)


## Run Batch

Run this only after the single-subject notebook succeeds. It calls `alice.load_trf()` directly for each subject.


In [ ]:
if RUN_BATCH:
    for subject in subjects:
        print(f'Fitting sub-{subject}...')
        result = alice.load_trf(MODEL, subject=subject, **STATE, **TRF_OPTIONS)
        print(result)
else:
    print('RUN_BATCH is False; batch run was not launched.')
